In [ ]:
from arize.experimental.datasets import ArizeDatasetsClient
import os
from dotenv import load_dotenv

load_dotenv()

HOST = os.getenv("ARIZE_GRPC_HOST")
API_KEY = os.getenv("ARIZE_API_KEY")
SPACE_ID = os.getenv("ARIZE_SPACE_ID")
DATASET_ID = "RGF0YXNldDoxNToxZnFp" # taken from 02_dataset.ipynb

client = ArizeDatasetsClient(
    api_key=API_KEY,
    host=HOST
)

In [ ]:
df = client.get_dataset(space_id=SPACE_ID, dataset_id=DATASET_ID)
df

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# llm = ChatOpenAI(model="gpt-4.1-2025-04-14", temperature=0.0)
# chain = llm | StrOutputParser()

def sample_task(dataset_row) -> str:
  return dataset_row.get("label")
  # return chain.invoke(dataset_row.get("question"))

In [ ]:
sample_task(df.iloc[0])

In [ ]:
from typing import Any, Optional, Mapping
from arize.experimental.datasets.experiments.evaluators.base import (
    EvaluationResult,
    CodeEvaluator,
    JSONSerializable,
    TaskOutput
)

class SampleEvaluator(CodeEvaluator):
    def evaluate(
        self,
        *,
        dataset_row: Optional[Mapping[str, JSONSerializable]] = None,
        output: Optional[TaskOutput] = None,
        **_: Any,
    ) -> EvaluationResult:
        ground_truth = dataset_row.get("label") if dataset_row is not None else None
        task_output = output

        is_good = ground_truth == task_output
        label = "match expected result" if is_good else "does not match expected result"
        
        return EvaluationResult(
            score=float(is_good),
            label=label,
            explanation=f"ground truth: {ground_truth}, task output: {task_output}"
        )

In [ ]:
results_df = client.run_experiment(
    space_id=SPACE_ID, 
    dataset_id=DATASET_ID,
    task=sample_task, 
    evaluators=[SampleEvaluator()], #include your evaluation functions here 
    experiment_name="code-eval-2",
    concurrency=10,
    dry_run=False
)